# Road Damage Inspection System - Final Project Master Notebook

**Authors:** `[Member 1 name]`, `[Member 2 name]`, `[Member 3 name]`  
**Course:** Deep Learning and Computer Vision  
**Notebook status:** Member 1 implemented; Member 2, Member 3, and shared report sections scaffolded below.

## Project notebook map

1. Shared project configuration and reproducibility
2. **Member 1:** full RDD2022 data engineering and detection EDA - implemented in this version
3. **Member 2:** YOLO11n and RT-DETR-R18 training/evaluation - prepared TODO section
4. **Member 3:** Pothole Mix, semantic segmentation, severity scoring, and app - prepared TODO section
5. **Shared:** model operations, maintenance, conclusion, references, and presentation evidence - prepared TODO section

---

# Part A - Member 1: RDD2022 data engineering and exploratory data analysis

**Recommended runtime:** Google Colab **CPU / High-RAM**. This notebook does not train a model and does not benefit materially from a GPU.

This notebook completes the Member 1 detection-data scope:

1. Downloads the official RDD2022 release from Figshare (12.36 GB, CC BY 4.0).
2. Inventories all 47,420 official images across all six countries and both Chinese capture domains.
3. Parses the original Pascal VOC XML labels and audits schema, missing files, malformed boxes, corrupt images, exact duplicates, and near duplicates.
4. Retains the complete labeled pool; no random image subset is discarded.
5. Creates leakage-safe 70/15/15 train/validation/test splits. Near-duplicate groups stay in one split.
6. Produces report-ready EDA tables, histograms, bivariate plots, heatmaps, Q-Q plots, samples, and outlier panels.
7. Exports one shared dataset in both **YOLO** and **COCO** annotation formats so YOLO11 and the official RT-DETR implementation can use identical images and splits.
8. Creates a held-out-country manifest for the cross-country experiment (train/validation on Japan + India + Czech; test on United States).

Official sources:

- Dataset: https://doi.org/10.6084/m9.figshare.21431547
- Dataset paper: https://doi.org/10.1002/gdj3.260
- RDD2022 paper preprint: https://arxiv.org/abs/2209.08538
- Official RT-DETR implementation: https://github.com/lyuwenyu/RT-DETR

> Run the cells from top to bottom. The first complete run can take 40-90 minutes depending on download speed and Google Drive performance. The notebook saves the final prepared archive and small reports to Drive; later model notebooks should use the prepared archive rather than download RDD2022 again.

## 0. Configuration

No Kaggle account or API token is required. The official Figshare download is used by default.

If you already have the official ZIP in Google Drive, set `SOURCE_ZIP_OVERRIDE` to that file. If you already extracted the dataset, set `EXTRACTED_ROOT_OVERRIDE` to the directory containing the country folders.

In [ ]:
from __future__ import annotations

import hashlib
import json
import math
import os
import random
import shutil
import subprocess
import sys
import time
import warnings
import zipfile
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from xml.etree import ElementTree as ET

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yaml
from IPython.display import Markdown, display
from PIL import Image, ImageDraw, ImageOps
from scipy import stats
from tqdm.auto import tqdm

try:
    import cv2
except ImportError:
    cv2 = None

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 100)

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

# ---------- Edit only this block if needed ----------
SOURCE_DOMAINS = ["China_Drone", "China_MotorBike", "Czech", "India", "Japan", "Norway", "United_States"]
DOMAIN_TO_COUNTRY = {
    "China_Drone": "China", "China_MotorBike": "China", "Czech": "Czech",
    "India": "India", "Japan": "Japan", "Norway": "Norway", "United_States": "United_States",
}
COUNTRIES = ["China", "Czech", "India", "Japan", "Norway", "United_States"]
HELD_OUT_COUNTRY = "United_States"
USE_FULL_LABELED_DATA = True
RANDOM_SEED = 42
NEAR_DUP_HAMMING_THRESHOLD = 4  # 64-bit difference hash
QUALITY_WORKERS = min(8, os.cpu_count() or 2)
SAVE_PREPARED_ZIP = True

# Optional existing inputs. Leave as empty strings for the standard workflow.
SOURCE_ZIP_OVERRIDE = ""
EXTRACTED_ROOT_OVERRIDE = ""

OFFICIAL_URL = "https://ndownloader.figshare.com/files/38030910"
OFFICIAL_ZIP_BYTES = 13_264_172_619
OFFICIAL_ZIP_MD5 = "b62bd51d2ffcfaa76c60f234f0cc2bb3"
CLASS_TO_YOLO = {"D00": 0, "D10": 1, "D20": 2, "D40": 3}
CLASS_NAMES = {
    "D00": "Longitudinal crack",
    "D10": "Transverse crack",
    "D20": "Alligator crack",
    "D40": "Pothole",
}

if IN_COLAB:
    LOCAL_ROOT = Path("/content/member1_rdd2022_work")
    PERSIST_ROOT = Path("/content/drive/MyDrive/RDD2022_Project/member1_outputs")
else:
    LOCAL_ROOT = Path.cwd() / "member1_rdd2022_work"
    PERSIST_ROOT = Path.cwd() / "member1_outputs"

ZIP_PATH = Path(SOURCE_ZIP_OVERRIDE) if SOURCE_ZIP_OVERRIDE else LOCAL_ROOT / "RDD2022_official.zip"
RAW_ROOT = Path(EXTRACTED_ROOT_OVERRIDE) if EXTRACTED_ROOT_OVERRIDE else LOCAL_ROOT / "raw"
PREPARED_ROOT = LOCAL_ROOT / "rdd2022_yolo_coco_full_labeled"
REPORTS_DIR = PREPARED_ROOT / "reports"
FIGURES_DIR = PREPARED_ROOT / "figures"
ANNOTATIONS_DIR = PREPARED_ROOT / "annotations"

for directory in [LOCAL_ROOT, PERSIST_ROOT, REPORTS_DIR, FIGURES_DIR, ANNOTATIONS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print(f"Runtime: {'Google Colab' if IN_COLAB else 'local Python'}")
print(f"Local workspace: {LOCAL_ROOT}")
print(f"Persistent outputs: {PERSIST_ROOT}")
print(f"Source domains: {SOURCE_DOMAINS}")
print("Selection policy: retain every usable labeled image (no random subset)")

## 1. Download and selectively extract the official data

Every official image is extracted. RDD2022's official test folders do not contain public ground-truth XML labels, so the notebook inventories and analyzes their image properties but cannot use them for supervised evaluation. New train/validation/test partitions are made from the complete labeled training pool.

In [ ]:
def download_official_zip(url: str, destination: Path) -> None:
    def md5_file(path: Path, chunk_size: int = 8 * 1024 * 1024) -> str:
        digest = hashlib.md5()
        with path.open("rb") as stream:
            for chunk in iter(lambda: stream.read(chunk_size), b""):
                digest.update(chunk)
        return digest.hexdigest()

    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists() and destination.stat().st_size == OFFICIAL_ZIP_BYTES:
        print(f"Verifying existing ZIP checksum: {destination}")
        if md5_file(destination) != OFFICIAL_ZIP_MD5:
            raise RuntimeError("Existing ZIP has the correct size but the wrong official MD5 checksum.")
        print(f"Using verified existing ZIP: {destination}")
        return
    if destination.exists():
        print(f"Resuming partial download at {destination.stat().st_size / 1e9:.2f} GB")
    command = ["wget", "-c", "--show-progress", "-O", str(destination), url]
    subprocess.run(command, check=True)
    actual = destination.stat().st_size
    if actual != OFFICIAL_ZIP_BYTES:
        raise RuntimeError(f"Download size mismatch: expected {OFFICIAL_ZIP_BYTES}, got {actual}")
    if md5_file(destination) != OFFICIAL_ZIP_MD5:
        raise RuntimeError("Downloaded ZIP failed the official Figshare MD5 checksum.")


def is_selected_member(member_name: str) -> bool:
    parts = Path(member_name.replace("\\", "/")).parts
    if not any(domain in parts for domain in SOURCE_DOMAINS):
        return False
    is_image = "images" in parts and ("train" in parts or "test" in parts) and Path(member_name).suffix.lower() in {".jpg", ".jpeg", ".png"}
    is_xml = "train" in parts and "xmls" in parts and Path(member_name).suffix.lower() == ".xml"
    return is_image or is_xml


def find_domain_assets(root: Path, domain: str) -> tuple[Path, Path, Path | None]:
    image_dirs = [p for p in root.rglob("images") if p.parent.name == "train" and domain in p.parts]
    test_dirs = [p for p in root.rglob("images") if p.parent.name == "test" and domain in p.parts]
    xml_dirs = [p for p in root.rglob("xmls") if p.parent.name == "annotations" and p.parent.parent.name == "train" and domain in p.parts]
    if len(image_dirs) != 1 or len(xml_dirs) != 1:
        raise FileNotFoundError(
            f"Expected one labeled image directory and one XML directory for {domain}; "
            f"found images={image_dirs}, xmls={xml_dirs}"
        )
    if len(test_dirs) > 1:
        raise FileNotFoundError(f"Found multiple official test directories for {domain}: {test_dirs}")
    return image_dirs[0], xml_dirs[0], (test_dirs[0] if test_dirs else None)


if not EXTRACTED_ROOT_OVERRIDE:
    download_official_zip(OFFICIAL_URL, ZIP_PATH)
    extraction_marker = RAW_ROOT / ".all_domains_complete"
    if not extraction_marker.exists():
        print("Opening official nested ZIP and extracting all domain images/XML files...")
        RAW_ROOT.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(ZIP_PATH) as outer_archive:
            outer_by_domain = {}
            for member in outer_archive.infolist():
                member_path = Path(member.filename.replace("\\", "/"))
                if member_path.suffix.lower() == ".zip" and member_path.stem in SOURCE_DOMAINS:
                    outer_by_domain[member_path.stem] = member
            missing_domains = sorted(set(SOURCE_DOMAINS) - set(outer_by_domain))
            if missing_domains:
                raise FileNotFoundError(f"Missing nested domain ZIPs: {missing_domains}")

            for domain in SOURCE_DOMAINS:
                print(f"Extracting nested archive: {domain}.zip")
                with outer_archive.open(outer_by_domain[domain]) as nested_stream:
                    with zipfile.ZipFile(nested_stream) as nested_archive:
                        members = [m for m in nested_archive.infolist() if is_selected_member(m.filename)]
                        for member in tqdm(members, desc=f"Extracting {domain}"):
                            nested_archive.extract(member, RAW_ROOT / "RDD2022")
        extraction_marker.touch()
    else:
        print("All-domain extraction already complete.")

domain_assets = {domain: find_domain_assets(RAW_ROOT, domain) for domain in SOURCE_DOMAINS}
for domain, (image_dir, xml_dir, test_dir) in domain_assets.items():
    print(f"{domain:16s} labeled={image_dir} | XML={xml_dir} | official test={test_dir}")

## 2. Parse Pascal VOC XML annotations and build the raw schema

The unit of analysis is separated into two tables:

- `image_manifest_raw`: one row per labeled source image.
- `boxes_raw`: one row per annotated damage instance.

This is preferable to forcing variable-length bounding boxes into a single wide table.

In [ ]:
def child_text(node: ET.Element | None, path: str, default: str = "") -> str:
    if node is None:
        return default
    child = node.find(path)
    return child.text.strip() if child is not None and child.text else default


def parse_domain(source_domain: str, image_dir: Path, xml_dir: Path):
    country = DOMAIN_TO_COUNTRY[source_domain]
    image_paths = [p for p in image_dir.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"}]
    image_by_stem = {p.stem: p for p in image_paths}
    xml_paths = sorted(xml_dir.glob("*.xml"))
    xml_by_stem = {p.stem: p for p in xml_paths}

    manifest_rows, box_rows, parse_errors = [], [], []
    all_stems = sorted(set(image_by_stem) | set(xml_by_stem))

    for stem in tqdm(all_stems, desc=f"Parsing {country}", leave=False):
        image_path = image_by_stem.get(stem)
        xml_path = xml_by_stem.get(stem)
        image_id = f"{source_domain}__{stem}"
        width = height = depth = np.nan
        objects = []
        xml_status = "missing_xml" if xml_path is None else "ok"

        if xml_path is not None:
            try:
                root = ET.parse(xml_path).getroot()
                size = root.find("size")
                width = int(child_text(size, "width", "0")) or np.nan
                height = int(child_text(size, "height", "0")) or np.nan
                depth = int(child_text(size, "depth", "0")) or np.nan
                for obj_idx, obj in enumerate(root.findall("object")):
                    label = child_text(obj, "name").upper()
                    bbox = obj.find("bndbox")
                    try:
                        xmin = float(child_text(bbox, "xmin"))
                        ymin = float(child_text(bbox, "ymin"))
                        xmax = float(child_text(bbox, "xmax"))
                        ymax = float(child_text(bbox, "ymax"))
                    except (TypeError, ValueError):
                        xmin = ymin = xmax = ymax = np.nan
                    objects.append(label)
                    box_rows.append({
                        "image_id": image_id,
                        "country": country,
                        "source_domain": source_domain,
                        "source_stem": stem,
                        "object_index": obj_idx,
                        "class_code": label,
                        "xmin": xmin,
                        "ymin": ymin,
                        "xmax": xmax,
                        "ymax": ymax,
                    })
            except Exception as exc:
                xml_status = "parse_error"
                parse_errors.append({"image_id": image_id, "xml_path": str(xml_path), "error": repr(exc)})

        known_objects = [label for label in objects if label in CLASS_TO_YOLO]
        unknown_objects = [label for label in objects if label not in CLASS_TO_YOLO]
        signature = "+".join(sorted(set(known_objects), key=lambda x: CLASS_TO_YOLO[x])) or "none"
        manifest_rows.append({
            "image_id": image_id,
            "country": country,
            "source_domain": source_domain,
            "official_split": "train_labeled",
            "source_stem": stem,
            "image_path": str(image_path) if image_path else "",
            "xml_path": str(xml_path) if xml_path else "",
            "image_exists": image_path is not None,
            "xml_status": xml_status,
            "xml_width": width,
            "xml_height": height,
            "xml_depth": depth,
            "annotation_count": len(known_objects),
            "unknown_annotation_count": len(unknown_objects),
            "class_signature": signature,
            **{f"has_{code}": int(code in known_objects) for code in CLASS_TO_YOLO},
        })
    return manifest_rows, box_rows, parse_errors


all_manifest, all_boxes, all_parse_errors = [], [], []
for source_domain, (image_dir, xml_dir, _) in domain_assets.items():
    manifest_rows, box_rows, parse_errors = parse_domain(source_domain, image_dir, xml_dir)
    all_manifest.extend(manifest_rows)
    all_boxes.extend(box_rows)
    all_parse_errors.extend(parse_errors)

image_manifest_raw = pd.DataFrame(all_manifest)
boxes_raw = pd.DataFrame(all_boxes)
parse_errors_df = pd.DataFrame(all_parse_errors)

unlabeled_rows = []
for source_domain, (_, _, test_dir) in domain_assets.items():
    if test_dir is None:
        continue
    country = DOMAIN_TO_COUNTRY[source_domain]
    for image_path in sorted(p for p in test_dir.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"}):
        image_id = f"{source_domain}__official_test__{image_path.stem}"
        unlabeled_rows.append({
            "image_id": image_id, "country": country, "source_domain": source_domain,
            "official_split": "test_unlabeled", "source_stem": image_path.stem,
            "image_path": str(image_path), "xml_path": "", "image_exists": True,
            "xml_status": "not_released", "xml_width": np.nan, "xml_height": np.nan,
            "xml_depth": np.nan, "annotation_count": np.nan, "unknown_annotation_count": np.nan,
            "class_signature": "unlabeled", **{f"has_{code}": 0 for code in CLASS_TO_YOLO},
        })
unlabeled_inventory = pd.DataFrame(unlabeled_rows)
full_official_inventory = pd.concat([image_manifest_raw, unlabeled_inventory], ignore_index=True, sort=False)

if boxes_raw.empty:
    raise RuntimeError("No annotations were parsed. Check the dataset directory structure.")

boxes_raw["is_known_class"] = boxes_raw["class_code"].isin(CLASS_TO_YOLO)
print(f"Raw image records: {len(image_manifest_raw):,}")
print(f"Official all-image inventory: {len(full_official_inventory):,}")
if len(full_official_inventory) != 47_420:
    print("WARNING: official inventory does not equal the published 47,420 images; inspect the extracted release.")
print(f"Raw bounding-box records: {len(boxes_raw):,}")
display(image_manifest_raw.head())
display(boxes_raw.head())

In [ ]:
raw_schema = pd.DataFrame({
    "table": ["image_manifest_raw"] * len(image_manifest_raw.columns) + ["boxes_raw"] * len(boxes_raw.columns),
    "field": list(image_manifest_raw.columns) + list(boxes_raw.columns),
    "dtype": [str(t) for t in image_manifest_raw.dtypes] + [str(t) for t in boxes_raw.dtypes],
    "missing_values": image_manifest_raw.isna().sum().tolist() + boxes_raw.isna().sum().tolist(),
})
raw_counts = pd.DataFrame({
    "metric": [
        "official_all_images", "official_labeled_images", "official_test_images_without_public_xml",
        "bounding_boxes", "missing_image_files", "missing_xml_files",
        "xml_parse_errors", "images_without_damage", "unknown_class_boxes"
    ],
    "value": [
        len(full_official_inventory), len(image_manifest_raw), len(unlabeled_inventory),
        len(boxes_raw), (~image_manifest_raw.image_exists).sum(),
        (image_manifest_raw.xml_status == "missing_xml").sum(),
        (image_manifest_raw.xml_status == "parse_error").sum(),
        (image_manifest_raw.annotation_count == 0).sum(), (~boxes_raw.is_known_class).sum(),
    ],
})
display(raw_counts)
display(raw_schema)

## 3. Complete labeled-pool selection

Every usable image with a released XML annotation is retained. Natural country and class imbalance is documented rather than hidden through undersampling. The official XML files also contain codes outside the four project targets (for example D44, D50, and REPAIR); these boxes are fully audited but not remapped into D00/D10/D20/D40. Images with no target-class box remain in the dataset as target-task negatives. No entire image is removed merely because it also contains a non-target annotation.

In [ ]:
def even_capacity_quotas(capacity: pd.Series, target: int) -> dict[str, int]:
    capacity = capacity.astype(int).copy()
    quotas = pd.Series(0, index=capacity.index, dtype=int)
    remaining = min(target, int(capacity.sum()))
    while remaining > 0:
        active = [key for key in capacity.index if quotas[key] < capacity[key]]
        if not active:
            break
        share = max(1, remaining // len(active))
        progressed = 0
        for key in sorted(active):
            add = min(share, int(capacity[key] - quotas[key]), remaining)
            quotas[key] += add
            remaining -= add
            progressed += add
            if remaining == 0:
                break
        if progressed == 0:
            break
    return quotas.to_dict()


def proportional_stratified_take(frame: pd.DataFrame, n: int, stratum_col: str, seed: int) -> pd.DataFrame:
    if n >= len(frame):
        return frame.copy()
    counts = frame[stratum_col].value_counts().sort_index()
    ideal = counts / counts.sum() * n
    allocation = np.floor(ideal).astype(int).clip(upper=counts)
    remaining = n - int(allocation.sum())
    order = (ideal - allocation).sort_values(ascending=False).index.tolist()
    cursor = 0
    while remaining > 0:
        key = order[cursor % len(order)]
        if allocation[key] < counts[key]:
            allocation[key] += 1
            remaining -= 1
        cursor += 1
    pieces = []
    for offset, key in enumerate(counts.index):
        group = frame[frame[stratum_col] == key]
        pieces.append(group.sample(n=int(allocation[key]), random_state=seed + offset))
    return pd.concat(pieces, ignore_index=True).sample(frac=1, random_state=seed).reset_index(drop=True)


candidates = image_manifest_raw[
    image_manifest_raw.image_exists
    & image_manifest_raw.xml_status.eq("ok")
].copy()

selected = candidates.sort_values("image_id").reset_index(drop=True)
print(f"Complete usable labeled pool before image decoding: {len(selected):,}")
display(selected.groupby(["source_domain", "country", "class_signature"]).size().rename("images").reset_index())

## 4. Image integrity, visual features, and duplicate audit

Brightness, contrast, and blur are calculated on a small grayscale thumbnail for speed. Exact duplicates use SHA-1 of the original file. Near duplicates use a 64-bit difference hash and locality-sensitive banding; the result is used as a leakage-control group, not as a reason to silently delete data.

In [ ]:
def sha1_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha1()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def difference_hash(gray: Image.Image, hash_size: int = 8) -> str:
    resized = gray.resize((hash_size + 1, hash_size), Image.Resampling.LANCZOS)
    arr = np.asarray(resized, dtype=np.int16)
    bits = arr[:, 1:] > arr[:, :-1]
    value = 0
    for bit in bits.ravel():
        value = (value << 1) | int(bit)
    return f"{value:0{hash_size * hash_size // 4}x}"


def scan_image(row: dict) -> dict:
    path = Path(row["image_path"])
    result = {"image_id": row["image_id"], "is_corrupt": False, "scan_error": ""}
    try:
        with Image.open(path) as opened:
            opened.load()
            image = ImageOps.exif_transpose(opened).convert("RGB")
        width, height = image.size
        gray = image.convert("L")
        thumb = gray.copy()
        thumb.thumbnail((256, 256), Image.Resampling.LANCZOS)
        arr = np.asarray(thumb, dtype=np.float32)
        if cv2 is not None:
            blur_score = float(cv2.Laplacian(arr.astype(np.uint8), cv2.CV_64F).var())
        else:
            gy, gx = np.gradient(arr)
            blur_score = float(np.var(gx) + np.var(gy))
        result.update({
            "actual_width": width,
            "actual_height": height,
            "aspect_ratio": width / height if height else np.nan,
            "brightness_mean": float(arr.mean()),
            "contrast_std": float(arr.std()),
            "blur_laplacian_var": blur_score,
            "sha1": sha1_file(path),
            "dhash64": difference_hash(gray),
            "file_bytes": path.stat().st_size,
        })
    except Exception as exc:
        result.update({
            "is_corrupt": True,
            "scan_error": repr(exc),
            "actual_width": np.nan,
            "actual_height": np.nan,
            "aspect_ratio": np.nan,
            "brightness_mean": np.nan,
            "contrast_std": np.nan,
            "blur_laplacian_var": np.nan,
            "sha1": "",
            "dhash64": "",
            "file_bytes": path.stat().st_size if path.exists() else 0,
        })
    return result


scan_started = time.time()
records = full_official_inventory[["image_id", "image_path"]].to_dict("records")
with ThreadPoolExecutor(max_workers=QUALITY_WORKERS) as pool:
    scanned_rows = list(tqdm(pool.map(scan_image, records), total=len(records), desc="Scanning images"))
image_quality = pd.DataFrame(scanned_rows)
full_inventory_scanned = full_official_inventory.merge(image_quality, on="image_id", how="left", validate="one_to_one")
selected_scanned = selected.merge(image_quality, on="image_id", how="left", validate="one_to_one")
selected_scanned["resolution_mpx"] = selected_scanned.actual_width * selected_scanned.actual_height / 1e6
selected_scanned["xml_size_mismatch"] = (
    selected_scanned.xml_width.notna()
    & selected_scanned.xml_height.notna()
    & ((selected_scanned.xml_width != selected_scanned.actual_width) | (selected_scanned.xml_height != selected_scanned.actual_height))
)
print(f"Image scan finished in {(time.time() - scan_started) / 60:.1f} minutes")
full_inventory_scanned["resolution_mpx"] = full_inventory_scanned.actual_width * full_inventory_scanned.actual_height / 1e6
print(f"Corrupt images across the official release: {full_inventory_scanned.is_corrupt.sum():,}")
print(f"XML/image size mismatches: {selected_scanned.xml_size_mismatch.sum():,}")

In [ ]:
class UnionFind:
    def __init__(self, n: int):
        self.parent = list(range(n))
        self.rank = [0] * n

    def find(self, x: int) -> int:
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, a: int, b: int) -> None:
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return
        if self.rank[ra] < self.rank[rb]:
            ra, rb = rb, ra
        self.parent[rb] = ra
        if self.rank[ra] == self.rank[rb]:
            self.rank[ra] += 1


def hamming_hex(a: str, b: str) -> int:
    return (int(a, 16) ^ int(b, 16)).bit_count()


clean = full_inventory_scanned[~full_inventory_scanned.is_corrupt].reset_index(drop=True).copy()
uf = UnionFind(len(clean))

# Exact duplicate connections.
for _, indices in clean.groupby("sha1").groups.items():
    indices = list(indices)
    for idx in indices[1:]:
        uf.union(indices[0], idx)

# Exact-radius search in a BK-tree over 64-bit dHashes. This scales better than all-pairs comparison
# and does not miss pairs at the configured Hamming distance.
bk_values: list[int] = []
bk_indices: list[list[int]] = []
bk_children: list[dict[int, int]] = []

def bk_query(value: int, radius: int) -> list[int]:
    if not bk_values:
        return []
    matches, stack = [], [0]
    while stack:
        node = stack.pop()
        distance = (value ^ bk_values[node]).bit_count()
        if distance <= radius:
            matches.extend(bk_indices[node])
        low, high = distance - radius, distance + radius
        stack.extend(child for edge, child in bk_children[node].items() if low <= edge <= high)
    return matches

def bk_insert(value: int, row_index: int) -> None:
    if not bk_values:
        bk_values.append(value); bk_indices.append([row_index]); bk_children.append({})
        return
    node = 0
    while True:
        distance = (value ^ bk_values[node]).bit_count()
        if distance == 0:
            bk_indices[node].append(row_index)
            return
        if distance in bk_children[node]:
            node = bk_children[node][distance]
        else:
            new_node = len(bk_values)
            bk_children[node][distance] = new_node
            bk_values.append(value); bk_indices.append([row_index]); bk_children.append({})
            return

for idx, value_hex in enumerate(tqdm(clean.dhash64, desc="Near-duplicate BK-tree")):
    value = int(value_hex, 16)
    for previous_idx in bk_query(value, NEAR_DUP_HAMMING_THRESHOLD):
        uf.union(previous_idx, idx)
    bk_insert(value, idx)

roots = [uf.find(i) for i in range(len(clean))]
root_to_group = {root: f"dupgroup_{number:05d}" for number, root in enumerate(sorted(set(roots)))}
clean["near_duplicate_group"] = [root_to_group[root] for root in roots]
clean["exact_duplicate_count"] = clean.groupby("sha1").sha1.transform("size")
clean["near_duplicate_group_size"] = clean.groupby("near_duplicate_group").image_id.transform("size")

exact_duplicate_rows = clean[clean.exact_duplicate_count > 1].copy()
near_duplicate_rows = clean[clean.near_duplicate_group_size > 1].copy()

# Retain every usable labeled image, including exact duplicates, to honor the full-data policy.
# Exact and near duplicates are grouped into one split so they cannot leak across partitions.
final_images = clean[clean.official_split == "train_labeled"].sort_values("image_id").reset_index(drop=True)
print(f"Exact duplicate rows found: {len(exact_duplicate_rows):,}")
print(f"Near-duplicate rows (including exact duplicates): {len(near_duplicate_rows):,}")
print(f"Usable labeled images retained: {len(final_images):,}")

## 5. Bounding-box quality checks

Boxes outside the image are reported. For export, coordinates are clipped to the image boundary and degenerate boxes are excluded. No issue is silently hidden: every excluded or adjusted box remains present in the audit table.

In [ ]:
selected_boxes = boxes_raw[
    boxes_raw.image_id.isin(final_images.image_id) & boxes_raw.is_known_class
].copy()
dimensions = final_images[["image_id", "actual_width", "actual_height"]]
selected_boxes = selected_boxes.merge(dimensions, on="image_id", how="left", validate="many_to_one")

selected_boxes["nonfinite_box"] = ~np.isfinite(selected_boxes[["xmin", "ymin", "xmax", "ymax"]]).all(axis=1)
selected_boxes["degenerate_box"] = (
    (selected_boxes.xmax <= selected_boxes.xmin) | (selected_boxes.ymax <= selected_boxes.ymin)
)
selected_boxes["out_of_bounds"] = (
    (selected_boxes.xmin < 0) | (selected_boxes.ymin < 0)
    | (selected_boxes.xmax > selected_boxes.actual_width)
    | (selected_boxes.ymax > selected_boxes.actual_height)
)

selected_boxes["clip_xmin"] = selected_boxes.xmin.clip(lower=0)
selected_boxes["clip_ymin"] = selected_boxes.ymin.clip(lower=0)
selected_boxes["clip_xmax"] = np.minimum(selected_boxes.xmax, selected_boxes.actual_width)
selected_boxes["clip_ymax"] = np.minimum(selected_boxes.ymax, selected_boxes.actual_height)
selected_boxes["box_width"] = selected_boxes.clip_xmax - selected_boxes.clip_xmin
selected_boxes["box_height"] = selected_boxes.clip_ymax - selected_boxes.clip_ymin
selected_boxes["box_area"] = selected_boxes.box_width * selected_boxes.box_height
selected_boxes["relative_width"] = selected_boxes.box_width / selected_boxes.actual_width
selected_boxes["relative_height"] = selected_boxes.box_height / selected_boxes.actual_height
selected_boxes["relative_area"] = selected_boxes.box_area / (
    selected_boxes.actual_width * selected_boxes.actual_height
)
selected_boxes["center_x"] = (selected_boxes.clip_xmin + selected_boxes.clip_xmax) / 2 / selected_boxes.actual_width
selected_boxes["center_y"] = (selected_boxes.clip_ymin + selected_boxes.clip_ymax) / 2 / selected_boxes.actual_height
selected_boxes["extremely_small_box"] = selected_boxes.relative_area < 0.0001
selected_boxes["export_valid"] = (
    ~selected_boxes.nonfinite_box
    & (selected_boxes.box_width >= 1)
    & (selected_boxes.box_height >= 1)
    & (selected_boxes.relative_area > 0)
)

bbox_quality_summary = pd.DataFrame({
    "issue": ["nonfinite_box", "degenerate_box", "out_of_bounds", "extremely_small_box", "excluded_from_export"],
    "count": [
        selected_boxes.nonfinite_box.sum(), selected_boxes.degenerate_box.sum(),
        selected_boxes.out_of_bounds.sum(), selected_boxes.extremely_small_box.sum(),
        (~selected_boxes.export_valid).sum(),
    ],
})
display(bbox_quality_summary)

## 6. Exploratory data analysis

The figures below cover the course rubric's image-data requirements and the proposal's detection EDA checklist: record counts, schema/missingness, raw samples, class/country balance, damages per image, multi-class images, bounding-box geometry, relative size, object-center heatmap, resolution, brightness, contrast, blur, bivariate comparisons, Q-Q plot, and outliers.

In [ ]:
def save_figure(name: str) -> None:
    plt.savefig(FIGURES_DIR / name, dpi=180, bbox_inches="tight")
    plt.show()
    plt.close()


eda_images = final_images.copy()
eda_boxes = selected_boxes[selected_boxes.export_valid].copy()
eda_boxes["class_name"] = eda_boxes.class_code.map(CLASS_NAMES)

# Summary table.
summary_metrics = pd.DataFrame({
    "metric": [
        "official_all_images", "prepared_labeled_images", "target_bounding_boxes", "images_without_target_damage", "multi_target_class_images",
        "corrupt_selected_images", "exact_duplicate_rows", "near_duplicate_rows",
        "out_of_bounds_boxes", "extremely_small_boxes"
    ],
    "value": [
        len(full_inventory_scanned), len(eda_images), len(eda_boxes), (eda_images.annotation_count == 0).sum(),
        (eda_images.class_signature.str.contains(r"\+")).sum(), selected_scanned.is_corrupt.sum(),
        len(exact_duplicate_rows), len(near_duplicate_rows), selected_boxes.out_of_bounds.sum(),
        selected_boxes.extremely_small_box.sum(),
    ],
})
display(summary_metrics)

raw_code_counts = boxes_raw.class_code.value_counts().rename_axis("class_code").reset_index(name="boxes")
raw_code_counts["mapping"] = np.where(raw_code_counts.class_code.isin(CLASS_TO_YOLO), "target class", "audited / not exported")
display(raw_code_counts)
plt.figure(figsize=(12, 5))
palette = raw_code_counts.mapping.map({"target class": "#2678B2", "audited / not exported": "#9E9E9E"})
plt.bar(raw_code_counts.class_code, raw_code_counts.boxes, color=palette)
plt.title("All annotation codes in the official XML files")
plt.xlabel("Annotation code")
plt.ylabel("Bounding boxes")
plt.xticks(rotation=35, ha="right")
save_figure("00_all_annotation_code_counts.png")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.countplot(data=eda_images, x="country", order=COUNTRIES, ax=axes[0], color="#2678B2")
axes[0].set_title("Images by country")
axes[0].tick_params(axis="x", rotation=20)
class_order = list(CLASS_TO_YOLO)
sns.countplot(data=eda_boxes, x="class_code", order=class_order, ax=axes[1], color="#E6862A")
axes[1].set_title("Bounding boxes by damage class")
axes[1].set_xlabel("Class code")
save_figure("01_counts_country_and_class.png")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
max_count = max(1, int(eda_images.annotation_count.quantile(0.99)))
sns.histplot(eda_images.annotation_count.clip(upper=max_count), discrete=True, ax=axes[0], color="#2678B2")
axes[0].set_title("Damages per image (clipped at 99th percentile)")
country_class = pd.crosstab(eda_boxes.country, eda_boxes.class_code).reindex(index=COUNTRIES, columns=class_order, fill_value=0)
sns.heatmap(country_class, annot=True, fmt=",", cmap="YlOrRd", ax=axes[1])
axes[1].set_title("Damage-class instances by country")
save_figure("02_damage_counts_and_country_class_heatmap.png")

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
sns.histplot(eda_boxes.relative_width, bins=50, ax=axes[0], color="#2678B2")
sns.histplot(eda_boxes.relative_height, bins=50, ax=axes[1], color="#43A047")
sns.histplot(eda_boxes.relative_area.clip(lower=1e-8), bins=60, log_scale=True, ax=axes[2], color="#E6862A")
axes[0].set_title("Relative box width")
axes[1].set_title("Relative box height")
axes[2].set_title("Relative box area (log x-axis)")
save_figure("03_bounding_box_distributions.png")

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
plot_boxes = eda_boxes.copy()
plot_boxes["log10_relative_area"] = np.log10(plot_boxes.relative_area.clip(lower=1e-8))
sns.violinplot(data=plot_boxes, x="class_code", y="log10_relative_area", order=class_order, inner="quartile", ax=axes[0])
axes[0].set_title("Relative box area by class")
heatmap, x_edges, y_edges = np.histogram2d(eda_boxes.center_x, eda_boxes.center_y, bins=30, range=[[0, 1], [0, 1]])
axes[1].imshow(heatmap.T, origin="upper", extent=[0, 1, 1, 0], cmap="magma", aspect="auto")
axes[1].set_title("Object-center heatmap")
axes[1].set_xlabel("Normalized x")
axes[1].set_ylabel("Normalized y")
save_figure("04_box_area_violin_and_center_heatmap.png")

eda_all_images = full_inventory_scanned[~full_inventory_scanned.is_corrupt].copy()
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
sns.scatterplot(data=eda_all_images, x="actual_width", y="actual_height", hue="country", alpha=0.35, s=18, ax=axes[0, 0])
axes[0, 0].set_title("Image resolution by country")
sns.histplot(data=eda_all_images, x="brightness_mean", hue="country", bins=45, element="step", stat="density", common_norm=False, ax=axes[0, 1])
axes[0, 1].set_title("Brightness distribution")
sns.histplot(data=eda_all_images, x="contrast_std", bins=50, ax=axes[1, 0], color="#43A047")
axes[1, 0].set_title("Contrast distribution")
sns.histplot(data=eda_all_images, x="blur_laplacian_var", bins=50, log_scale=(True, False), ax=axes[1, 1], color="#8E5EB7")
axes[1, 1].set_title("Blur score distribution (log x-axis)")
save_figure("05_resolution_brightness_contrast_blur.png")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
merged_counts = eda_images[["image_id", "brightness_mean", "annotation_count"]]
axes[0].hexbin(merged_counts.brightness_mean, merged_counts.annotation_count, gridsize=35, mincnt=1, cmap="viridis")
axes[0].set_xlabel("Mean brightness")
axes[0].set_ylabel("Damage instances")
axes[0].set_title("Brightness vs. damages per image")
stats.probplot(eda_images.brightness_mean.dropna(), dist="norm", plot=axes[1])
axes[1].set_title("Q-Q plot of image brightness")
save_figure("06_bivariate_brightness_and_qq.png")

In [ ]:
def draw_annotated_image(image_row: pd.Series, title: str | None = None) -> Image.Image:
    image = ImageOps.exif_transpose(Image.open(image_row.image_path)).convert("RGB")
    draw = ImageDraw.Draw(image)
    palette = {"D00": "#00E5FF", "D10": "#FFEA00", "D20": "#FF6D00", "D40": "#D500F9"}
    rows = eda_boxes[eda_boxes.image_id == image_row.image_id]
    line_width = max(2, round(min(image.size) / 250))
    for box in rows.itertuples():
        draw.rectangle([box.clip_xmin, box.clip_ymin, box.clip_xmax, box.clip_ymax], outline=palette[box.class_code], width=line_width)
        draw.text((box.clip_xmin + 3, box.clip_ymin + 3), box.class_code, fill=palette[box.class_code], stroke_width=1, stroke_fill="black")
    return image


def plot_image_rows(rows: pd.DataFrame, filename: str, heading: str, columns: int = 4) -> None:
    rows = rows.reset_index(drop=True)
    nrows = math.ceil(len(rows) / columns)
    fig, axes = plt.subplots(nrows, columns, figsize=(4 * columns, 3.4 * nrows))
    axes = np.atleast_1d(axes).ravel()
    for ax, (_, row) in zip(axes, rows.iterrows()):
        image = draw_annotated_image(row)
        ax.imshow(image)
        ax.set_title(f"{row.country} | {row.image_id}\n{row.annotation_count} boxes", fontsize=9)
        ax.axis("off")
    for ax in axes[len(rows):]:
        ax.axis("off")
    fig.suptitle(heading, fontsize=15, y=1.01)
    plt.tight_layout()
    save_figure(filename)


sample_rows = []
for offset, class_code in enumerate(CLASS_TO_YOLO):
    ids = eda_boxes.loc[eda_boxes.class_code == class_code, "image_id"].drop_duplicates()
    chosen = ids.sample(n=min(3, len(ids)), random_state=RANDOM_SEED + offset)
    sample_rows.append(eda_images[eda_images.image_id.isin(chosen)])
class_samples = pd.concat(sample_rows).drop_duplicates("image_id").head(12)
plot_image_rows(class_samples, "07_annotated_class_samples.png", "Annotated samples across damage classes")

# Distribution-based outliers: 3 darkest, 3 brightest, 3 blurriest, 3 sharpest.
outlier_rows = pd.concat([
    eda_images.nsmallest(3, "brightness_mean"),
    eda_images.nlargest(3, "brightness_mean"),
    eda_images.nsmallest(3, "blur_laplacian_var"),
    eda_images.nlargest(3, "blur_laplacian_var"),
]).drop_duplicates("image_id").head(12)
plot_image_rows(outlier_rows, "08_visual_outliers.png", "Visual outliers: lighting and sharpness")

## 7. Leakage-safe primary and cross-country splits

All models for the main champion-challenger comparison must use these same splits. Using fewer RT-DETR images than YOLO would confound architecture with data quantity and is therefore not done here.

In [ ]:
def stratified_group_sample(group_table: pd.DataFrame, n_groups: int, seed: int) -> set[str]:
    if n_groups <= 0:
        return set()
    table = group_table.copy()
    table["stratum"] = table.source_domain + "|" + table.class_signature
    sampled = proportional_stratified_take(table, min(n_groups, len(table)), "stratum", seed)
    return set(sampled.near_duplicate_group)


group_table = (
    final_images.groupby("near_duplicate_group", as_index=False)
    .agg(
        country=("country", "first"), source_domain=("source_domain", "first"),
        class_signature=("class_signature", "first"), group_size=("image_id", "size")
    )
)
test_groups = stratified_group_sample(group_table, round(0.15 * len(group_table)), RANDOM_SEED + 501)
remaining_groups = group_table[~group_table.near_duplicate_group.isin(test_groups)]
val_groups = stratified_group_sample(remaining_groups, round((0.15 / 0.85) * len(remaining_groups)), RANDOM_SEED + 502)

def primary_split(group_id: str) -> str:
    if group_id in test_groups:
        return "test"
    if group_id in val_groups:
        return "val"
    return "train"

final_images["split"] = final_images.near_duplicate_group.map(primary_split)
selected_boxes = selected_boxes.merge(final_images[["image_id", "split"]], on="image_id", how="inner", validate="many_to_one")

split_summary = (
    final_images.groupby(["split", "country"]).size().rename("images").reset_index()
)
split_leakage = final_images.groupby("near_duplicate_group").split.nunique().max()
assert split_leakage == 1, "Near-duplicate leakage detected across primary splits"
display(pd.crosstab(final_images.split, final_images.country, margins=True))
display(pd.crosstab(final_images.split, final_images.class_signature, margins=True))

# Cross-country split: held-out country is test; remaining countries get an 85/15 train/validation split.
development_groups = group_table[group_table.country != HELD_OUT_COUNTRY].copy()
cross_val_groups = stratified_group_sample(development_groups, round(0.15 * len(development_groups)), RANDOM_SEED + 601)
heldout_groups = set(final_images.loc[final_images.country == HELD_OUT_COUNTRY, "near_duplicate_group"])

def cross_country_split(row: pd.Series) -> str:
    if row.country == HELD_OUT_COUNTRY:
        return "test"
    # If a visual near-duplicate of a held-out image occurs in another country, exclude the
    # development-side copy from this experiment instead of leaking it into train/validation.
    if row.near_duplicate_group in heldout_groups:
        return "exclude_leakage"
    if row.near_duplicate_group in cross_val_groups:
        return "val"
    return "train"

final_images["cross_country_split"] = final_images.apply(cross_country_split, axis=1)
included_cross = final_images[final_images.cross_country_split != "exclude_leakage"]
assert included_cross.groupby("near_duplicate_group").cross_country_split.nunique().max() == 1
display(pd.crosstab(final_images.cross_country_split, final_images.country, margins=True))

## 8. Export shared YOLO and COCO datasets

- YOLO class IDs: `0..3`.
- COCO category IDs: `1..4`.
- Both formats point to the same copied image files and the same split assignments.
- Empty YOLO label files are retained for valid no-damage images.

In [ ]:
# Rebuild generated dataset folders so stale files cannot contaminate the export.
for generated in [PREPARED_ROOT / "images", PREPARED_ROOT / "labels"]:
    if generated.exists():
        shutil.rmtree(generated)
for split in ["train", "val", "test"]:
    (PREPARED_ROOT / "images" / split).mkdir(parents=True, exist_ok=True)
    (PREPARED_ROOT / "labels" / split).mkdir(parents=True, exist_ok=True)

final_images["export_stem"] = final_images.image_id.str.replace(r"[^A-Za-z0-9_.-]", "_", regex=True)
final_images["export_filename"] = final_images.export_stem + final_images.image_path.map(lambda p: Path(p).suffix.lower())

boxes_for_export = selected_boxes[selected_boxes.export_valid].copy()
boxes_for_export = boxes_for_export.merge(
    final_images[["image_id", "export_stem", "export_filename", "split"]],
    on=["image_id", "split"], how="inner", validate="many_to_one"
)
boxes_by_image = {image_id: group for image_id, group in boxes_for_export.groupby("image_id", sort=False)}

for row in tqdm(final_images.itertuples(), total=len(final_images), desc="Copying images and writing YOLO labels"):
    destination = PREPARED_ROOT / "images" / row.split / row.export_filename
    try:
        os.link(row.image_path, destination)  # avoids a second local copy on the default Colab/local filesystem
    except OSError:
        shutil.copy2(row.image_path, destination)
    image_boxes = boxes_by_image.get(row.image_id)
    label_lines = []
    for box in (() if image_boxes is None else image_boxes.itertuples()):
        x_center = (box.clip_xmin + box.clip_xmax) / 2 / box.actual_width
        y_center = (box.clip_ymin + box.clip_ymax) / 2 / box.actual_height
        width = box.box_width / box.actual_width
        height = box.box_height / box.actual_height
        label_lines.append(
            f"{CLASS_TO_YOLO[box.class_code]} {x_center:.8f} {y_center:.8f} {width:.8f} {height:.8f}"
        )
    label_path = PREPARED_ROOT / "labels" / row.split / f"{row.export_stem}.txt"
    label_path.write_text("\n".join(label_lines) + ("\n" if label_lines else ""), encoding="utf-8")

yolo_yaml = {
    "path": "/content/rdd2022_yolo_coco_full_labeled",
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": {index: f"{code} - {CLASS_NAMES[code]}" for code, index in CLASS_TO_YOLO.items()},
}
(PREPARED_ROOT / "dataset.yaml").write_text(yaml.safe_dump(yolo_yaml, sort_keys=False), encoding="utf-8")


def build_coco_json(split_column: str, split_value: str) -> dict:
    image_rows = final_images[final_images[split_column] == split_value].sort_values("image_id").reset_index(drop=True)
    id_map = {image_id: idx + 1 for idx, image_id in enumerate(image_rows.image_id)}
    coco_images = [
        {
            "id": id_map[row.image_id],
            "file_name": f"{row.split}/{row.export_filename}",
            "width": int(row.actual_width),
            "height": int(row.actual_height),
            "country": row.country,
            "source_domain": row.source_domain,
        }
        for row in image_rows.itertuples()
    ]
    valid_boxes = boxes_for_export[boxes_for_export.image_id.isin(id_map)].sort_values(["image_id", "object_index"])
    coco_annotations = []
    for annotation_id, box in enumerate(valid_boxes.itertuples(), start=1):
        coco_annotations.append({
            "id": annotation_id,
            "image_id": id_map[box.image_id],
            "category_id": CLASS_TO_YOLO[box.class_code] + 1,
            "bbox": [float(box.clip_xmin), float(box.clip_ymin), float(box.box_width), float(box.box_height)],
            "area": float(box.box_area),
            "iscrowd": 0,
        })
    categories = [
        {"id": index + 1, "name": code, "supercategory": "road_damage", "description": CLASS_NAMES[code]}
        for code, index in CLASS_TO_YOLO.items()
    ]
    return {"images": coco_images, "annotations": coco_annotations, "categories": categories}


for split in ["train", "val", "test"]:
    main_coco = build_coco_json("split", split)
    (ANNOTATIONS_DIR / f"instances_{split}.json").write_text(json.dumps(main_coco, indent=2), encoding="utf-8")
    cross_coco = build_coco_json("cross_country_split", split)
    (ANNOTATIONS_DIR / f"instances_cross_country_{split}.json").write_text(json.dumps(cross_coco, indent=2), encoding="utf-8")

print("Exported image and label counts")
for split in ["train", "val", "test"]:
    image_count = len(list((PREPARED_ROOT / "images" / split).iterdir()))
    label_count = len(list((PREPARED_ROOT / "labels" / split).iterdir()))
    print(f"{split:5s}: images={image_count:,}, YOLO labels={label_count:,}")

## 9. Save audit tables, a report-ready summary, and the prepared archive

The generated Markdown summary contains actual values from this run. Copy its tables and selected plots into the final report, but add interpretation in your own words.

In [ ]:
# Small, human-readable audit artifacts.
raw_schema.to_csv(REPORTS_DIR / "raw_schema_and_missingness.csv", index=False)
raw_counts.to_csv(REPORTS_DIR / "raw_record_counts.csv", index=False)
raw_code_counts.to_csv(REPORTS_DIR / "all_annotation_code_counts.csv", index=False)
image_manifest_raw.to_csv(REPORTS_DIR / "raw_image_manifest.csv", index=False)
full_inventory_scanned.to_csv(REPORTS_DIR / "official_all_image_inventory_scanned.csv", index=False)
final_images.to_csv(REPORTS_DIR / "prepared_image_manifest.csv", index=False)
selected_boxes.to_csv(REPORTS_DIR / "prepared_bbox_audit.csv", index=False)
bbox_quality_summary.to_csv(REPORTS_DIR / "bbox_quality_summary.csv", index=False)
exact_duplicate_rows.to_csv(REPORTS_DIR / "exact_duplicate_rows.csv", index=False)
near_duplicate_rows.to_csv(REPORTS_DIR / "near_duplicate_rows.csv", index=False)
split_summary.to_csv(REPORTS_DIR / "primary_split_summary.csv", index=False)
parse_errors_df.to_csv(REPORTS_DIR / "xml_parse_errors.csv", index=False)

run_config = {
    "official_url": OFFICIAL_URL,
    "official_dataset_doi": "10.6084/m9.figshare.21431547",
    "dataset_license": "CC BY 4.0",
    "countries": COUNTRIES,
    "held_out_country": HELD_OUT_COUNTRY,
    "selection_policy": "all usable labeled images; no random subset",
    "random_seed": RANDOM_SEED,
    "near_duplicate_hamming_threshold": NEAR_DUP_HAMMING_THRESHOLD,
    "class_to_yolo": CLASS_TO_YOLO,
    "final_image_count": len(final_images),
    "final_valid_box_count": int(selected_boxes.export_valid.sum()),
}
(REPORTS_DIR / "run_config.json").write_text(json.dumps(run_config, indent=2), encoding="utf-8")

class_counts = eda_boxes.class_code.value_counts().reindex(CLASS_TO_YOLO).fillna(0).astype(int)
country_counts = final_images.country.value_counts().reindex(COUNTRIES).fillna(0).astype(int)
source_domain_counts = final_images.source_domain.value_counts().reindex(SOURCE_DOMAINS).fillna(0).astype(int)
split_counts = final_images.split.value_counts().reindex(["train", "val", "test"]).fillna(0).astype(int)

report_lines = [
    "# RDD2022 Member 1 - Data and EDA summary",
    "",
    f"Generated with random seed `{RANDOM_SEED}` from the official RDD2022 release.",
    "",
    "## Dataset summary",
    "",
    f"- Official images inventoried: **{len(full_inventory_scanned):,}**",
    f"- Official test images without public XML labels: **{len(unlabeled_inventory):,}**",
    f"- Prepared labeled images: **{len(final_images):,}**",
    f"- Valid exported bounding boxes: **{int(selected_boxes.export_valid.sum()):,}**",
    f"- Non-target annotation boxes audited but not exported: **{int((~boxes_raw.is_known_class).sum()):,}**",
    f"- Images without annotated damage: **{int((final_images.annotation_count == 0).sum()):,}**",
    f"- Images containing multiple damage classes: **{int(final_images.class_signature.str.contains(r'\+').sum()):,}**",
    f"- Corrupt selected images: **{int(selected_scanned.is_corrupt.sum()):,}**",
    f"- Exact duplicate rows found: **{len(exact_duplicate_rows):,}**",
    f"- Near-duplicate rows grouped for leakage control: **{len(near_duplicate_rows):,}**",
    f"- Out-of-bounds boxes (clipped for export): **{int(selected_boxes.out_of_bounds.sum()):,}**",
    f"- Boxes excluded from export: **{int((~selected_boxes.export_valid).sum()):,}**",
    "",
    "## Images by country",
    "",
    country_counts.rename("images").to_frame().to_markdown(),
    "",
    "## Labeled images by source domain",
    "",
    source_domain_counts.rename("images").to_frame().to_markdown(),
    "",
    "## Bounding boxes by class",
    "",
    class_counts.rename("boxes").to_frame().to_markdown(),
    "",
    "## All official XML annotation codes",
    "",
    raw_code_counts.to_markdown(index=False),
    "",
    "## Primary split",
    "",
    split_counts.rename("images").to_frame().to_markdown(),
    "",
    "## Interpretation prompts for the final report",
    "",
    "1. Explain the country and class imbalances and why mAP/per-class recall are more informative than accuracy alone.",
    "2. Discuss whether small relative box areas make thin cracks a small-object problem.",
    "3. Compare brightness/contrast distributions by country and connect domain shift to the held-out-country experiment.",
    "4. Describe exact/near-duplicate handling and why keeping each group in one split prevents leakage.",
    "5. Use the sample and outlier panels to discuss shadows, lane markings, blur, darkness, and annotation quality.",
    "",
    "## Sources",
    "",
    "- Arya et al., RDD2022 dataset: https://doi.org/10.6084/m9.figshare.21431547",
    "- Arya et al., dataset paper: https://doi.org/10.1002/gdj3.260",
]
report_path = REPORTS_DIR / "EDA_REPORT.md"
report_path.write_text("\n".join(report_lines), encoding="utf-8")

citation_bib = r'''@dataset{arya2022rdd2022_dataset,
  author    = {Arya, Deeksha and Maeda, Hiroya and Sekimoto, Yoshihide and others},
  title     = {RDD2022 - The multi-national Road Damage Dataset released through CRDDC'2022},
  year      = {2022},
  publisher = {figshare},
  doi       = {10.6084/m9.figshare.21431547}
}

@article{arya2024rdd2022,
  author  = {Arya, Deeksha and Maeda, Hiroya and Ghosh, Sanjay Kumar and Toshniwal, Durga and Sekimoto, Yoshihide},
  title   = {RDD2022: A multi-national image dataset for automatic road damage detection},
  journal = {Geoscience Data Journal},
  year    = {2024},
  doi     = {10.1002/gdj3.260}
}
'''
(PREPARED_ROOT / "CITATION.bib").write_text(citation_bib, encoding="utf-8")

dataset_readme = f'''# Prepared full labeled RDD2022 dataset

This folder was generated by the Member 1 data/EDA notebook using random seed {RANDOM_SEED}.

- `images/{{train,val,test}}`: shared images for both detection models
- `labels/{{train,val,test}}`: YOLO labels (class IDs 0-3)
- `annotations/instances_*.json`: COCO labels (category IDs 1-4) for the main split
- `annotations/instances_cross_country_*.json`: COCO labels for the held-out-country split
- `dataset.yaml`: Ultralytics data configuration (unzip this folder to `/content/rdd2022_yolo_coco_full_labeled`)
- `figures`: report-ready EDA plots
- `reports`: schema, manifests, quality audits, and an EDA summary

Project handoff:

- Member 2 must use these exact primary splits for both YOLO11n and RT-DETR-R18.
- Member 2 uses the `instances_cross_country_*.json` files for the US held-out experiment.
- Member 3 prepares the complete Pothole Mix dataset in the master notebook's Part C; it is a separate image-mask data contract.
- All members complete the master notebook's Part D after model evaluation.

Source: RDD2022, DOI 10.6084/m9.figshare.21431547, CC BY 4.0.
'''
(PREPARED_ROOT / "README.md").write_text(dataset_readme, encoding="utf-8")

# Copy small reports/figures to Drive even if the full archive is disabled.
small_output = PERSIST_ROOT / "reports_and_figures"
if small_output.exists():
    shutil.rmtree(small_output)
small_output.mkdir(parents=True, exist_ok=True)
shutil.copytree(REPORTS_DIR, small_output / "reports")
shutil.copytree(FIGURES_DIR, small_output / "figures")
shutil.copy2(PREPARED_ROOT / "CITATION.bib", small_output / "CITATION.bib")

archive_destination = None
if SAVE_PREPARED_ZIP:
    print("Creating the prepared full-labeled dataset archive locally. This can take several minutes...")
    local_archive = Path(shutil.make_archive(str(LOCAL_ROOT / "rdd2022_yolo_coco_full_labeled"), "zip", PREPARED_ROOT))
    archive_destination = PERSIST_ROOT / local_archive.name
    print(f"Copying {local_archive.stat().st_size / 1e9:.2f} GB archive to persistent storage...")
    shutil.copy2(local_archive, archive_destination)

display(Markdown(report_path.read_text(encoding="utf-8")))
print("\nDONE")
print(f"Reports and figures: {small_output}")
if archive_destination:
    print(f"Prepared YOLO + COCO archive: {archive_destination}")

## 10. Handoff to Members 2 and 3

Member 2 should unzip `rdd2022_yolo_coco_full_labeled.zip` into `/content/rdd2022_yolo_coco_full_labeled` in a **GPU** Colab session.

- YOLO11: use `/content/rdd2022_yolo_coco_full_labeled/dataset.yaml`.
- Official RT-DETR-R18: use the same image folders and `/content/rdd2022_yolo_coco_full_labeled/annotations/instances_{train,val,test}.json`.
- Do not resample separately for the two models; comparison requires identical train/validation/test data.
- For cross-country generalization, use the `instances_cross_country_*.json` files and report United States test metrics separately.

Member 3's Pothole Mix preparation and segmentation EDA are a separate data pipeline. They are intentionally not mixed into this detection notebook because the images, masks, quality checks, and leakage risks are different.

---

# Part B - Member 2: Object detection models and experiments

**Owner:** `[Member 2 name]`  
**Status:** TODO scaffold  
**Runtime:** Google Colab GPU  
**Required input from Part A:** the prepared full-labeled RDD2022 archive, shared primary split, and cross-country COCO manifests.

Member 2 must not create a new random split. YOLO11n and RT-DETR-R18 must use the same train/validation/test images. Save every result table with the random seed, data-manifest hash, package versions, checkpoint path, training duration, and GPU type.

## B1. GPU environment and data handoff

Expected paths after unzipping:

- `/content/rdd2022_yolo_coco_full_labeled/dataset.yaml`
- `/content/rdd2022_yolo_coco_full_labeled/annotations/instances_train.json`
- `/content/rdd2022_yolo_coco_full_labeled/annotations/instances_val.json`
- `/content/rdd2022_yolo_coco_full_labeled/annotations/instances_test.json`

In [ ]:
# TODO(Member 2): switch this notebook to a GPU runtime, mount Drive, and unzip the
# Member 1 prepared archive to /content/rdd2022_yolo_coco_full_labeled.
# Do not execute model training until the manifest and split counts match Part A.

## B2. Detection experiment registry

Fill one row immediately after every run.

| Run ID | Model | Data manifest/hash | Image size | Epochs | Batch | Seed | GPU | Best checkpoint | Train time | Notes |
|---|---|---|---:|---:|---:|---:|---|---|---|---|
| `det-yolo-main-01` | YOLO11n | TODO | 640 | TODO | TODO | 42 | TODO | TODO | TODO | TODO |
| `det-rtdetr-main-01` | RT-DETR-R18 | TODO | 640 | TODO | TODO | 42 | TODO | TODO | TODO | TODO |

## B3. YOLO11n - pipeline test, training, and resume

Required stages:

1. 300-500 images, 1-2 epochs: validate labels and metrics.
2. 1,000-2,000 images, 3-5 epochs: estimate time and memory.
3. Full labeled training split with early stopping and resumable `last` checkpoint.
4. Load only `best` for final test evaluation.

In [ ]:
# TODO(Member 2): install/pin Ultralytics, record versions, train YOLO11n on dataset.yaml,
# save best/last checkpoints to Drive, and export per-epoch metrics.

## B4. RT-DETR-R18 - pipeline test, training, and resume

Use the official `lyuwenyu/RT-DETR` implementation (or its verified compatible R18 checkpoint), not Ultralytics RT-DETR-L/X mislabeled as R18. Configure the COCO loader with the Part A image folders and `instances_*.json` files.

In [ ]:
# TODO(Member 2): clone a fixed official RT-DETR commit, record the commit hash,
# configure RT-DETR-R18 for four categories, run staged training, and save best/last checkpoints.

## B5. Detection champion-challenger evaluation

Populate the same test-set metrics for both models.

| Metric | YOLO11n | RT-DETR-R18 | Winner / interpretation |
|---|---:|---:|---|
| mAP@0.50 | TODO | TODO | TODO |
| mAP@0.50:0.95 | TODO | TODO | TODO |
| Precision | TODO | TODO | TODO |
| Recall | TODO | TODO | TODO |
| F1 | TODO | TODO | TODO |
| D00 AP | TODO | TODO | TODO |
| D10 AP | TODO | TODO | TODO |
| D20 AP | TODO | TODO | TODO |
| D40 AP / pothole recall | TODO | TODO | TODO |
| Inference ms/image | TODO | TODO | TODO |
| Parameters | TODO | TODO | TODO |
| Peak GPU memory | TODO | TODO | TODO |
| Training time | TODO | TODO | TODO |

Required figures: confusion matrix, precision-recall curves, per-class AP comparison, speed-accuracy plot, and matched qualitative predictions on the same images.

In [ ]:
# TODO(Member 2): load both best checkpoints, evaluate once on the untouched shared test set,
# create the comparison dataframe/figures, and save raw predictions for reproducibility.

## B6. Resolution experiment

Compare the selected champion at 512 vs 640 under a controlled budget. Keep every setting except image size fixed and report small-box/thin-crack recall alongside compute cost.

In [ ]:
# TODO(Member 2): controlled 512-vs-640 experiment and comparison table.

## B7. Cross-country generalization

Use `instances_cross_country_*.json`: United States is the held-out test country; all non-US development data uses the prepared train/validation assignment. Do not tune on US test labels.

In [ ]:
# TODO(Member 2): train the chosen detector without US development images, evaluate on US once,
# and compare per-class metrics with the in-domain primary test result.

## B8. Detection error analysis

Analyze false negatives and false positives by damage class, relative box size, country/source domain, brightness, blur, road markings, shadows, pavement joints, and multi-damage scenes. Link failures back to Part A EDA rather than showing only successful examples.

In [ ]:
# TODO(Member 2): merge saved predictions with the Part A image/bbox audit tables and generate
# matched failure panels plus quantitative error slices.

---

# Part C - Member 3: Semantic segmentation, severity, and application

**Owner:** `[Member 3 name]`  
**Status:** TODO scaffold  
**Runtime:** Google Colab GPU for training; CPU is sufficient for data EDA and app smoke tests.

Use all 4,340 Pothole Mix image-mask pairs with the documented 3,340/496/504 train/validation/test split unless a duplicate-leakage audit justifies a corrected group-aware split. DeepLabV3-MobileNetV3 and SegFormer-B0 must use identical data and preprocessing.

## C1. Pothole Mix acquisition, schema, and license record

In [ ]:
# TODO(Member 3): add the verified official download/source, component-dataset licenses,
# checksums, directory discovery, image-mask pairing, and split inventory.

## C2. Segmentation data quality and EDA

Required outputs: pair counts, missing/corrupt files, image-mask dimension alignment, mask colors/classes, empty masks, damage-pixel percentage, connected components, small-mask outliers, duplicate/near-duplicate leakage, resolution/brightness distributions, and aligned image-mask samples.

In [ ]:
# TODO(Member 3): full Pothole Mix data audit and report-ready segmentation EDA.

## C3. DeepLabV3-MobileNetV3

Run the same staged validation workflow as Part B. Record preprocessing, augmentations, BCE + Dice implementation, checkpoint criteria, and all package versions.

In [ ]:
# TODO(Member 3): DeepLabV3-MobileNetV3 pipeline test, full training, resume, and best-checkpoint evaluation.

## C4. SegFormer-B0

Use the identical full split, resize/crop policy, class mapping, loss definition, and evaluation protocol used for DeepLabV3.

In [ ]:
# TODO(Member 3): SegFormer-B0 pipeline test, full training, resume, and best-checkpoint evaluation.

## C5. Segmentation champion-challenger evaluation

| Metric | DeepLabV3-MobileNetV3 | SegFormer-B0 | Winner / interpretation |
|---|---:|---:|---|
| IoU / mIoU | TODO | TODO | TODO |
| Dice | TODO | TODO | TODO |
| Pixel precision | TODO | TODO | TODO |
| Pixel recall | TODO | TODO | TODO |
| Pixel F1 | TODO | TODO | TODO |
| Boundary F1 | TODO | TODO | TODO |
| Inference ms/image | TODO | TODO | TODO |
| Parameters | TODO | TODO | TODO |
| Peak GPU memory | TODO | TODO | TODO |
| Training time | TODO | TODO | TODO |

In [ ]:
# TODO(Member 3): create the shared-test comparison, boundary-focused failure analysis,
# and matched image/mask/prediction panels.

## C6. Prototype severity and maintenance priority

Define a transparent, non-certified prototype rule using segmentation damage-area ratio, detection counts/classes, largest/total box area, and confidence. Calibrate thresholds only on development data and clearly state that the result is not a civil-engineering severity standard.

In [ ]:
# TODO(Member 3): implement severity features, documented thresholds, unit tests, and example cases.

## C7. Gradio or Streamlit application

Required display: uploaded image, detection boxes/classes/confidences, segmentation mask, damaged-area percentage, prototype priority, model/version metadata, latency, and a limitations notice.

In [ ]:
# TODO(Member 3): application implementation and CPU/GPU smoke-test evidence.

---

# Part D - Shared report, model operations, and conclusion

**Owners:** all three members  
**Status:** TODO scaffold

## D1. Final abstract

`[Write last: problem, full datasets, four models, core metrics/results, deployment prototype, and main conclusion in one concise paragraph.]`

## D2. Consolidated model scorecard

| Problem | Model | Primary quality metric | Key safety/recall metric | Latency | Parameters | Decision |
|---|---|---:|---:|---:|---:|---|
| Detection | YOLO11n | TODO | TODO | TODO | TODO | TODO |
| Detection | RT-DETR-R18 | TODO | TODO | TODO | TODO | TODO |
| Segmentation | DeepLabV3-MobileNetV3 | TODO | TODO | TODO | TODO | TODO |
| Segmentation | SegFormer-B0 | TODO | TODO | TODO | TODO | TODO |

## D3. Deployment architecture

`[Insert final architecture: upload/API -> validation/preprocessing -> detector -> conditional segmenter -> severity rule -> dashboard -> logging/review queue. Include model registry, monitoring, and rollback.]`

## D4. Model maintenance and parameter-update process

`[Define data/quality drift checks, per-country and per-class recall monitoring, low-confidence/high-severity review queues, annotation QA, retraining trigger, champion-challenger gate, versioning, rollback, and update approval.]`

## D5. Limitations, ethics, and intended use

`[Discuss country/domain bias, daylight/weather limitations, camera variation, annotation uncertainty, false-negative safety risk, non-certified severity, privacy, and the need for human inspection.]`

## D6. Conclusion

`[Summarize evidence-based findings, architecture trade-offs, cross-country result, resolution result, operational recommendation, limitations, and realistic next work.]`

## D7. References and reused code

Start with the generated `CITATION.bib`, then add model papers, official implementations, framework documentation, Pothole Mix/SHREC sources, and every reused code pattern. Record exact repository commit hashes where possible.

## D8. Presentation evidence

Suggested 20-minute structure: 2 minutes shared opening/problem; 5 minutes Member 1 data/EDA; 5 minutes Member 2 detection; 5 minutes Member 3 segmentation/app; 3 minutes shared comparison, operations, limitations, and conclusion.

In [ ]:
# TODO(All members): add final report-table exports, architecture figure path,
# presentation figure manifest, author contributions, and final reproducibility checklist.